<a href="https://colab.research.google.com/github/magenta/mt3/blob/main/mt3/colab/music_transcription_with_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music Transcription with Transformers

This notebook is an interactive demo of a few [music transcription models](g.co/magenta/mt3) created by Google's [Magenta](g.co/magenta) team.  You can upload audio and have one of our models automatically transcribe it.

<img src="https://magenta.tensorflow.org/assets/transcription-with-transformers/architecture_diagram.png" alt="Transformer-based transcription architecture">

The notebook supports two pre-trained models:
1. the piano transcription model from [our ISMIR 2021 paper](https://archives.ismir.net/ismir2021/paper/000030.pdf)
1. the multi-instrument transcription model from [our ICLR 2022 paper](https://openreview.net/pdf?id=iMSjopcOn0p)

**Caveat**: neither model is trained on singing.  If you upload audio with vocals, you will likely get weird results.  Multi-instrument transcription is still not a completely-solved problem and so you may get weird results regardless.

In any case, we hope you have fun transcribing!  Feel free to tweet any interesting output at [@GoogleMagenta](https://twitter.com/googlemagenta)...

### Instructions for running:

* Make sure to use a GPU runtime, click:  __Runtime >> Change Runtime Type >> GPU__
* Press ▶️ on the left of each cell to execute the cell
* In the __Load Model__ cell, choose either `ismir2021` for piano transcription or `mt3` for multi-instrument transcription
* In the __Upload Audio__ cell, choose an MP3 or WAV file from your computer when prompted
* Transcribe the audio using the __Transcribe Audio__ cell (it may take a few minutes depending on the length of the audio)

---

This notebook sends basic usage data to Google Analytics.  For more information, see [Google's privacy policy](https://policies.google.com/privacy).

In [ ]:
from google.colab import auth
auth.authenticate_user()


In [ ]:
# Copyright 2021 Google LLC. All Rights Reserved.

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#     http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

#@title Setup Environment
#@markdown Install MT3 and its dependencies (may take a few minutes).

!apt-get update -qq && apt-get install -qq libfluidsynth3 build-essential libasound2-dev libjack-dev

# install mt3
!git clone --branch=main https://github.com/magenta/mt3
!mv mt3 mt3_tmp; mv mt3_tmp/* .; rm -r mt3_tmp
!python3 -m pip install jax[cuda12] nest-asyncio pyfluidsynth==1.3.0 -e . -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# copy checkpoints
!gcloud storage cp --recursive gs://mt3/checkpoints .

# copy soundfont (originally from https://sites.google.com/site/soundfonts4u)
!gcloud storage cp gs://magentadata/soundfonts/SGM-v2.01-Sal-Guit-Bass-V1.3.sf2 .

# установка для постобработки
!pip install pretty_midi mido

import json
import IPython

# The below functions (load_gtag and log_event) handle Google Analytics event
# logging. The logging is anonymous and stores only very basic statistics of the
# audio and transcription e.g. length of audio, number of transcribed notes.

def load_gtag():
  """Loads gtag.js."""
  # Note: gtag.js MUST be loaded in the same cell execution as the one doing
  # synthesis. It does NOT persist across cell executions!
  html_code = '''
<!-- Global site tag (gtag.js) - Google Analytics -->
<script async src="https://www.googletagmanager.com/gtag/js?id=G-4P250YRJ08"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());
  gtag('config', 'G-4P250YRJ08',
       {'referrer': document.referrer.split('?')[0],
        'anonymize_ip': true,
        'page_title': '',
        'page_referrer': '',
        'cookie_prefix': 'magenta',
        'cookie_domain': 'auto',
        'cookie_expires': 0,
        'cookie_flags': 'SameSite=None;Secure'});
</script>
'''
  IPython.display.display(IPython.display.HTML(html_code))

def log_event(event_name, event_details):
  """Log event with name and details dictionary."""
  details_json = json.dumps(event_details)
  js_string = "gtag('event', '%s', %s);" % (event_name, details_json)
  IPython.display.display(IPython.display.Javascript(js_string))

load_gtag()
log_event('setupComplete', {})

In [ ]:
#@title Фикс конфликтов
!pip uninstall -y tensorflow tensorflow-text jax jaxlib t5x
!pip install tensorflow==2.19.0 tensorflow-text==2.19.0
!pip install jax==0.4.13 jaxlib==0.4.13
!pip install git+https://github.com/google-research/t5x@94a89731086a3ee35e6aba182ce10d109424011c
!pip install git+https://github.com/magenta/note-seq
!pip install demucs


In [ ]:
#@title Imports and Definitions

import functools
import os

import numpy as np
import tensorflow.compat.v2 as tf

import functools
import gin
import jax
import librosa
import note_seq
import seqio
import t5
import t5x

from mt3 import metrics_utils
from mt3 import models
from mt3 import network
from mt3 import note_sequences
from mt3 import preprocessors
from mt3 import spectrograms
from mt3 import vocabularies

from google.colab import files

import nest_asyncio
nest_asyncio.apply()

SAMPLE_RATE = 16000
SF2_PATH = 'SGM-v2.01-Sal-Guit-Bass-V1.3.sf2'

def upload_audio(sample_rate):
  data = list(files.upload().values())
  if len(data) > 1:
    print('Multiple files uploaded; using only one.')
  return note_seq.audio_io.wav_data_to_samples_librosa(
    data[0], sample_rate=sample_rate)



class InferenceModel(object):
  """Wrapper of T5X model for music transcription."""

  def __init__(self, checkpoint_path, model_type='mt3'):

    # Model Constants.
    if model_type == 'ismir2021':
      num_velocity_bins = 127
      self.encoding_spec = note_sequences.NoteEncodingSpec
      self.inputs_length = 512
    elif model_type == 'mt3':
      num_velocity_bins = 1
      self.encoding_spec = note_sequences.NoteEncodingWithTiesSpec
      self.inputs_length = 256
    else:
      raise ValueError('unknown model_type: %s' % model_type)

    gin_files = ['/content/mt3/gin/model.gin',
                 f'/content/mt3/gin/{model_type}.gin']

    self.batch_size = 8
    self.outputs_length = 1024
    self.sequence_length = {'inputs': self.inputs_length,
                            'targets': self.outputs_length}

    self.partitioner = t5x.partitioning.PjitPartitioner(
        num_partitions=1)

    # Build Codecs and Vocabularies.
    self.spectrogram_config = spectrograms.SpectrogramConfig()
    self.codec = vocabularies.build_codec(
        vocab_config=vocabularies.VocabularyConfig(
            num_velocity_bins=num_velocity_bins))
    self.vocabulary = vocabularies.vocabulary_from_codec(self.codec)
    self.output_features = {
        'inputs': seqio.ContinuousFeature(dtype=tf.float32, rank=2),
        'targets': seqio.Feature(vocabulary=self.vocabulary),
    }

    # Create a T5X model.
    self._parse_gin(gin_files)
    self.model = self._load_model()

    # Restore from checkpoint.
    self.restore_from_checkpoint(checkpoint_path)

  @property
  def input_shapes(self):
    return {
          'encoder_input_tokens': (self.batch_size, self.inputs_length),
          'decoder_input_tokens': (self.batch_size, self.outputs_length)
    }

  def _parse_gin(self, gin_files):
    """Parse gin files used to train the model."""
    gin_bindings = [
        'from __gin__ import dynamic_registration',
        'from mt3 import vocabularies',
        'VOCAB_CONFIG=@vocabularies.VocabularyConfig()',
        'vocabularies.VocabularyConfig.num_velocity_bins=%NUM_VELOCITY_BINS'
    ]
    with gin.unlock_config():
      gin.parse_config_files_and_bindings(
          gin_files, gin_bindings, finalize_config=False)

  def _load_model(self):
    """Load up a T5X `Model` after parsing training gin config."""
    model_config = gin.get_configurable(network.T5Config)()
    module = network.Transformer(config=model_config)
    return models.ContinuousInputsEncoderDecoderModel(
        module=module,
        input_vocabulary=self.output_features['inputs'].vocabulary,
        output_vocabulary=self.output_features['targets'].vocabulary,
        optimizer_def=t5x.adafactor.Adafactor(decay_rate=0.8, step_offset=0),
        input_depth=spectrograms.input_depth(self.spectrogram_config))


  def restore_from_checkpoint(self, checkpoint_path):
    """Restore training state from checkpoint, resets self._predict_fn()."""
    train_state_initializer = t5x.utils.TrainStateInitializer(
      optimizer_def=self.model.optimizer_def,
      init_fn=self.model.get_initial_variables,
      input_shapes=self.input_shapes,
      partitioner=self.partitioner)

    restore_checkpoint_cfg = t5x.utils.RestoreCheckpointConfig(
        path=checkpoint_path, mode='specific', dtype='float32')

    train_state_axes = train_state_initializer.train_state_axes
    self._predict_fn = self._get_predict_fn(train_state_axes)
    self._train_state = train_state_initializer.from_checkpoint_or_scratch(
        [restore_checkpoint_cfg], init_rng=jax.random.PRNGKey(0))

  @functools.lru_cache()
  def _get_predict_fn(self, train_state_axes):
    """Generate a partitioned prediction function for decoding."""
    def partial_predict_fn(params, batch, decode_rng):
      return self.model.predict_batch_with_aux(
          params, batch, decoder_params={'decode_rng': None})
    return self.partitioner.partition(
        partial_predict_fn,
        in_axis_resources=(
            train_state_axes.params,
            t5x.partitioning.PartitionSpec('data',), None),
        out_axis_resources=t5x.partitioning.PartitionSpec('data',)
    )

  def predict_tokens(self, batch, seed=0):
    """Predict tokens from preprocessed dataset batch."""
    prediction, _ = self._predict_fn(
        self._train_state.params, batch, jax.random.PRNGKey(seed))
    return self.vocabulary.decode_tf(prediction).numpy()

  def __call__(self, audio):
    """Infer note sequence from audio samples.

    Args:
      audio: 1-d numpy array of audio samples (16kHz) for a single example.

    Returns:
      A note_sequence of the transcribed audio.
    """
    ds = self.audio_to_dataset(audio)
    ds = self.preprocess(ds)

    model_ds = self.model.FEATURE_CONVERTER_CLS(pack=False)(
        ds, task_feature_lengths=self.sequence_length)
    model_ds = model_ds.batch(self.batch_size)

    inferences = (tokens for batch in model_ds.as_numpy_iterator()
                  for tokens in self.predict_tokens(batch))

    predictions = []
    for example, tokens in zip(ds.as_numpy_iterator(), inferences):
      predictions.append(self.postprocess(tokens, example))

    result = metrics_utils.event_predictions_to_ns(
        predictions, codec=self.codec, encoding_spec=self.encoding_spec)
    return result['est_ns']

  def audio_to_dataset(self, audio):
    """Create a TF Dataset of spectrograms from input audio."""
    frames, frame_times = self._audio_to_frames(audio)
    return tf.data.Dataset.from_tensors({
        'inputs': frames,
        'input_times': frame_times,
    })

  def _audio_to_frames(self, audio):
    """Compute spectrogram frames from audio."""
    frame_size = self.spectrogram_config.hop_width
    padding = [0, frame_size - len(audio) % frame_size]
    audio = np.pad(audio, padding, mode='constant')
    frames = spectrograms.split_audio(audio, self.spectrogram_config)
    num_frames = len(audio) // frame_size
    times = np.arange(num_frames) / self.spectrogram_config.frames_per_second
    return frames, times

  def preprocess(self, ds):
    pp_chain = [
        functools.partial(
            t5.data.preprocessors.split_tokens_to_inputs_length,
            sequence_length=self.sequence_length,
            output_features=self.output_features,
            feature_key='inputs',
            additional_feature_keys=['input_times']),
        # Cache occurs here during training.
        preprocessors.add_dummy_targets,
        functools.partial(
            preprocessors.compute_spectrograms,
            spectrogram_config=self.spectrogram_config)
    ]
    for pp in pp_chain:
      ds = pp(ds)
    return ds

  def postprocess(self, tokens, example):
    tokens = self._trim_eos(tokens)
    start_time = example['input_times'][0]
    # Round down to nearest symbolic token step.
    start_time -= start_time % (1 / self.codec.steps_per_second)
    return {
        'est_tokens': tokens,
        'start_time': start_time,
        # Internal MT3 code expects raw inputs, not used here.
        'raw_inputs': []
    }

  @staticmethod
  def _trim_eos(tokens):
    tokens = np.array(tokens, np.int32)
    if vocabularies.DECODED_EOS_ID in tokens:
      tokens = tokens[:np.argmax(tokens == vocabularies.DECODED_EOS_ID)]
    return tokens



In [ ]:
#@title Fix GPU/TPU usage
import t5x.partitioning

def fixed_bounds_from_last_device(last_device):
    if hasattr(last_device, 'coords') and len(last_device.coords) == 3:
        x, y, z = last_device.coords
        # Для GPU атрибут core_on_chip может отсутствовать, используем 1
        core = getattr(last_device, 'core_on_chip', 1)
        return x + 1, y + 1, z + 1, core
    else:
        # Для не-TPU платформ (GPU) возвращаем единичные размеры
        return 1, 1, 1, 1

t5x.partitioning.bounds_from_last_device = fixed_bounds_from_last_device


In [ ]:
#@title Load Model
#@markdown The `ismir2021` model transcribes piano only, with note velocities.
#@markdown The `mt3` model transcribes multiple simultaneous instruments,
#@markdown but without velocities.

MODEL = "mt3" #@param["ismir2021", "mt3"]

checkpoint_path = f'/content/checkpoints/{MODEL}/'

load_gtag()

log_event('loadModelStart', {'event_category': MODEL})
inference_model = InferenceModel(checkpoint_path, MODEL)
log_event('loadModelComplete', {'event_category': MODEL})


In [ ]:
#@title Postprocessing Settings (MIDI cleaning)

QUANTIZE_STEP_SEC = 0.05      #@param {type:"number"}  # quantization step (seconds)
MIN_DURATION_SEC = 0.03       #@param {type:"number"}  # minimum note duration
DEDUP_TOL_SEC = 0.02          #@param {type:"number"}  # duplicate onset tolerance
OVERLAP_TOL_SEC = 0.01        #@param {type:"number"}  # same-pitch overlap tolerance
DROP_EMPTY_INSTRUMENTS = True #@param {type:"boolean"}

print("✅ Postprocessing settings loaded")

In [ ]:
#@title Script Postprocessing Functions (MIDI cleaner)

import math
import tempfile
from dataclasses import dataclass, asdict
from pathlib import Path
import pretty_midi
import mido
import note_seq

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

@dataclass
class CleanStats:
    file: str
    load_backend: str
    symusic_backend: str | None = None
    instruments_total: int = 0
    instruments_dropped_empty: int = 0
    notes_before: int = 0
    notes_after: int = 0
    notes_removed_duplicate: int = 0
    notes_fixed_non_positive_duration: int = 0
    notes_extended_short: int = 0
    notes_trimmed_same_pitch_overlap: int = 0
    notes_removed_invalid: int = 0
    pitches_clamped: int = 0
    velocities_clamped: int = 0
    control_changes_clamped: int = 0
    control_changes_removed_duplicate: int = 0
    pitch_bends_clamped: int = 0
    quantized_events: int = 0
    read_error: str | None = None

def _quantize(x: float, step: float | None) -> float:
    if not step or step <= 0:
        return x
    return round(x / step) * step

def _clean_control_changes(inst, stats: CleanStats, tol: float = 1e-6):
    cleaned = []
    seen = set()
    for cc in sorted(inst.control_changes, key=lambda x: (float(x.time), int(x.number), int(x.value))):
        t = float(cc.time)
        number = clamp(int(cc.number), 0, 127)
        value = clamp(int(cc.value), 0, 127)
        if number != int(cc.number) or value != int(cc.value):
            stats.control_changes_clamped += 1
        if QUANTIZE_STEP_SEC:
            qt = _quantize(t, QUANTIZE_STEP_SEC)
            if abs(qt - t) > tol:
                stats.quantized_events += 1
            t = qt
        key = (round(t / max(tol, 1e-9)), number, value)
        if key in seen:
            stats.control_changes_removed_duplicate += 1
            continue
        seen.add(key)
        cleaned.append(pretty_midi.ControlChange(number=number, value=value, time=t))
    inst.control_changes = cleaned

def _clean_pitch_bends(inst, stats: CleanStats, tol: float = 1e-6):
    cleaned = []
    seen = set()
    for pb in sorted(inst.pitch_bends, key=lambda x: (float(x.time), int(x.pitch))):
        t = float(pb.time)
        pitch = clamp(int(pb.pitch), -8192, 8191)
        if pitch != int(pb.pitch):
            stats.pitch_bends_clamped += 1
        if QUANTIZE_STEP_SEC:
            qt = _quantize(t, QUANTIZE_STEP_SEC)
            if abs(qt - t) > tol:
                stats.quantized_events += 1
            t = qt
        key = (round(t / max(tol, 1e-9)), pitch)
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(pretty_midi.PitchBend(pitch=pitch, time=t))
    inst.pitch_bends = cleaned

def _prepare_note(note, stats: CleanStats):
    try:
        start = float(note.start)
        end = float(note.end)
        pitch = int(note.pitch)
        velocity = int(note.velocity)
    except Exception:
        stats.notes_removed_invalid += 1
        return None
    if any(map(math.isnan, [start, end])):
        stats.notes_removed_invalid += 1
        return None
    pitch2 = clamp(pitch, 0, 127)
    if pitch2 != pitch:
        stats.pitches_clamped += 1
        pitch = pitch2
    velocity2 = clamp(velocity, 1, 127)
    if velocity2 != velocity:
        stats.velocities_clamped += 1
        velocity = velocity2
    if QUANTIZE_STEP_SEC:
        start = _quantize(start, QUANTIZE_STEP_SEC)
        end = _quantize(end, QUANTIZE_STEP_SEC)
    if end <= start:
        end = start + MIN_DURATION_SEC
        stats.notes_fixed_non_positive_duration += 1
    if (end - start) < MIN_DURATION_SEC:
        end = start + MIN_DURATION_SEC
        stats.notes_extended_short += 1
    return {"start": start, "end": end, "pitch": pitch, "velocity": velocity}

def _deduplicate_notes(prepared_notes, stats: CleanStats):
    groups = {}
    for n in prepared_notes:
        onset_bucket = round(n["start"] / DEDUP_TOL_SEC)
        key = (n["pitch"], onset_bucket)
        prev = groups.get(key)
        if prev is None:
            groups[key] = n
        else:
            prev_score = (prev["end"] - prev["start"], prev["velocity"])
            cur_score = (n["end"] - n["start"], n["velocity"])
            if cur_score > prev_score:
                groups[key] = n
            stats.notes_removed_duplicate += 1
    return sorted(groups.values(), key=lambda x: (x["start"], x["pitch"], x["end"], x["velocity"]))

def _trim_same_pitch_overlaps(notes, stats: CleanStats, is_drum: bool):
    if is_drum:
        return notes
    cleaned = []
    last_idx_by_pitch = {}
    for n in notes:
        pitch = n["pitch"]
        if pitch in last_idx_by_pitch:
            prev = cleaned[last_idx_by_pitch[pitch]]
            if n["start"] < prev["end"] - OVERLAP_TOL_SEC:
                new_prev_end = max(prev["start"] + MIN_DURATION_SEC, n["start"])
                if new_prev_end < prev["end"] - OVERLAP_TOL_SEC:
                    prev["end"] = new_prev_end
                    stats.notes_trimmed_same_pitch_overlap += 1
                elif n["end"] <= prev["end"] + OVERLAP_TOL_SEC:
                    stats.notes_removed_invalid += 1
                    continue
        cleaned.append(n)
        last_idx_by_pitch[pitch] = len(cleaned) - 1
    return cleaned

def clean_pretty_midi(pm: pretty_midi.PrettyMIDI, file_name: str, load_backend: str, symusic_backend: str | None):
    stats = CleanStats(file=file_name, load_backend=load_backend, symusic_backend=symusic_backend)
    stats.instruments_total = len(pm.instruments)
    for inst in pm.instruments:
        stats.notes_before += len(inst.notes)
        prepared = []
        for note in sorted(inst.notes, key=lambda x: (float(x.start), int(x.pitch), float(x.end), int(x.velocity))):
            p = _prepare_note(note, stats)
            if p is not None:
                prepared.append(p)
        prepared = _deduplicate_notes(prepared, stats)
        prepared = _trim_same_pitch_overlaps(prepared, stats, is_drum=inst.is_drum)
        inst.notes = [pretty_midi.Note(velocity=n["velocity"], pitch=n["pitch"], start=n["start"], end=n["end"]) for n in prepared]
        _clean_control_changes(inst, stats)
        _clean_pitch_bends(inst, stats)
    if DROP_EMPTY_INSTRUMENTS:
        kept = [inst for inst in pm.instruments if inst.notes or inst.control_changes or inst.pitch_bends]
        stats.instruments_dropped_empty = len(pm.instruments) - len(kept)
        pm.instruments = kept
    stats.notes_after = sum(len(inst.notes) for inst in pm.instruments)
    return pm, stats

def load_pretty_midi_best_effort(path):
    errors = []
    try:
        return pretty_midi.PrettyMIDI(str(path)), "pretty_midi_direct", errors
    except Exception as e:
        errors.append(f"pretty_midi_direct: {type(e).__name__}: {e}")
    try:
        midi = mido.MidiFile(str(path), clip=True)
        with tempfile.NamedTemporaryFile(suffix=".mid", delete=False) as tmp:
            tmp_path = Path(tmp.name)
        midi.save(str(tmp_path))
        pm = pretty_midi.PrettyMIDI(str(tmp_path))
        tmp_path.unlink(missing_ok=True)
        return pm, "mido_clip_then_pretty_midi", errors
    except Exception as e:
        errors.append(f"mido_clip_then_pretty_midi: {type(e).__name__}: {e}")
    raise RuntimeError("Failed to read MIDI: " + " | ".join(errors))

def note_sequence_to_pretty_midi(ns):
    pm = pretty_midi.PrettyMIDI()
    instrument_map = {}
    for note in ns.notes:
        key = (note.instrument, note.program)
        if key not in instrument_map:
            inst = pretty_midi.Instrument(program=note.program, is_drum=note.is_drum, name=f"Instrument {note.instrument}")
            instrument_map[key] = inst
            pm.instruments.append(inst)
        inst = instrument_map[key]
        inst.notes.append(pretty_midi.Note(velocity=note.velocity, pitch=note.pitch, start=note.start_time, end=note.end_time))
    for inst in pm.instruments:
        inst.notes.sort(key=lambda x: x.start)
    return pm

def pretty_midi_to_note_sequence(pm, original_ns=None):
    ns = note_seq.NoteSequence()
    if original_ns:
        ns.CopyFrom(original_ns)
        del ns.notes[:]
    for inst in pm.instruments:
        for note in inst.notes:
            ns.notes.add(pitch=note.pitch, velocity=note.velocity, start_time=note.start, end_time=note.end,
                         instrument=inst.program, program=inst.program, is_drum=inst.is_drum)
    ns.total_time = max((note.end_time for note in ns.notes), default=0)
    return ns

def clean_note_sequence(ns):
    """Apply script postprocessing to a NoteSequence."""
    pm = note_sequence_to_pretty_midi(ns)
    pm, stats = clean_pretty_midi(pm, "transcribed", load_backend="note_sequence", symusic_backend=None)
    cleaned_ns = pretty_midi_to_note_sequence(pm, ns)
    return cleaned_ns, stats

print("✅ Cleaner functions loaded")

In [ ]:
#@title Upload Audio

import os
import librosa
import subprocess
from google.colab import files

load_gtag()
log_event('uploadAudioStart', {})

# audio = upload_audio(sample_rate=SAMPLE_RATE)
# log_event('uploadAudioComplete', {'value': round(len(audio) / SAMPLE_RATE)})
# note_seq.notebook_utils.colab_play(audio, sample_rate=SAMPLE_RATE)

# Upload the file
print("Please upload an MP3 or WAV file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

In [ ]:
#@title Run Demucs

print(f"\nRunning Demucs on '{file_name}'...")
print("Applying model and shift-averaging...")

model = "htdemucs"
subprocess.run([
    "demucs",
    "-n", model,
    "--two-stems=vocals",
    "--shifts=2",
    "--overlap=0.5",
    file_name
])

In [ ]:
#@title Start audio transcription
#@markdown This may take a few minutes depending on the length of the audio file
#@markdown you uploaded.

# Locate the output file in the new model folder
base_name = os.path.splitext(file_name)[0]
instrumental_path = f"separated/{model}/{base_name}/no_vocals.wav"

print(f"\nLoading track from {instrumental_path}...")

# Load the audio into MT3
audio, _ = librosa.load(instrumental_path, sr=SAMPLE_RATE)

log_event('uploadAudioComplete', {'value': round(len(audio) / SAMPLE_RATE)})

# Play the track to verify quality
note_seq.notebook_utils.colab_play(audio, sample_rate=SAMPLE_RATE)

load_gtag()

log_event('transcribeStart', {
    'event_category': MODEL,
    'value': round(len(audio) / SAMPLE_RATE)
})

# Run transcription
est_ns = inference_model(audio)

# log_event('transcribeComplete', {
#     'event_category': MODEL,
#     'value': round(len(audio) / SAMPLE_RATE),
#     'numNotes': sum(1 for note in est_ns.notes if not note.is_drum),
#     'numDrumNotes': sum(1 for note in est_ns.notes if note.is_drum),
#     'numPrograms': len(set(note.program for note in est_ns.notes
#                            if not note.is_drum))
# })

# note_seq.play_sequence(est_ns, synth=note_seq.fluidsynth,
#                        sample_rate=SAMPLE_RATE, sf2_path=SF2_PATH)
# note_seq.plot_sequence(est_ns)

In [ ]:
#@title Create cleaned notes (force piano)

cleaned_notes = []
for note in est_ns.notes:
    duration = note.end_time - note.start_time
    # Убираем короткие ноты и барабаны
    if duration >= 0.05 and not note.is_drum:
        note.program = 0   # фортепиано
        note.instrument = 0
        cleaned_notes.append(note)

# Заменяем ноты в est_ns
del est_ns.notes[:]
est_ns.notes.extend(cleaned_notes)

print(f"✅ Приведено к фортепиано, осталось нот: {len(est_ns.notes)}")

In [ ]:
#@title Apply Postprocessing and Listen

print("Applying script-based MIDI cleaning (quantization, dedup, overlap trim)...")
cleaned_ns, stats = clean_note_sequence(est_ns)

print("\n--- Cleaning Statistics ---")
print(f"Notes before cleaning: {stats.notes_before}")
print(f"Notes after cleaning:  {stats.notes_after}")
print(f"Removed duplicates: {stats.notes_removed_duplicate}")
print(f"Extended short notes: {stats.notes_extended_short}")
print(f"Trimmed overlapping same-pitch notes: {stats.notes_trimmed_same_pitch_overlap}")
print(f"Removed invalid notes: {stats.notes_removed_invalid}")

# Логируем финальное количество нот
log_event('transcribeComplete', {
    'event_category': MODEL,
    'value': round(len(audio) / SAMPLE_RATE),
    'numNotes': stats.notes_after,
    'numPrograms': 1  # так как всё приведено к пианино
})

print("\nListening to the cleaned piano transcription...")
note_seq.play_sequence(cleaned_ns, synth=note_seq.fluidsynth,
                       sample_rate=SAMPLE_RATE, sf2_path=SF2_PATH)
note_seq.plot_sequence(cleaned_ns)

In [ ]:
#@title Download MIDI Transcription

load_gtag()
log_event('downloadTranscription', {
    'event_category': MODEL,
    'value': round(len(audio) / SAMPLE_RATE),
    'numNotes': sum(1 for note in cleaned_ns.notes if not note.is_drum),
    'numDrumNotes': sum(1 for note in cleaned_ns.notes if note.is_drum),
    'numPrograms': len(set(note.program for note in cleaned_ns.notes if not note.is_drum))
})

note_seq.sequence_proto_to_midi_file(cleaned_ns, '/tmp/transcribed_cleaned.mid')
files.download('/tmp/transcribed_cleaned.mid')